# Wigner's friend — "when did it collapse?" has no operational answer

**The punchline.** Model a measurement honestly — as one system getting entangled with
another — and *from inside*, the measured qubit looks exactly, numerically, collapsed. Its
density matrix is indistinguishable from "the friend saw 0 or 1 and we don't know which".
From outside, the whole thing is still a unitary, and the outside observer can **undo it**
and recover interference that no collapse would have left behind.

Both descriptions are correct and neither is a mistake. The place where "the measurement
happened" is not fixed by the physics; it is a line *you* draw, and you can move it. Von
Neumann called it the cut.

Background: **[04 — Programs made of gates](../04-combinators.ipynb)** §7 (the tape,
`checkpoint`/`rewind`, and coherent records vs classical outcomes), and
**[quantum_eraser](quantum_eraser.ipynb)** — of which this notebook is the same experiment
with one qubit renamed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import qsim
from qsim import Circuit, QsimError
from qsim.decoherence import dephasing_coupling
from qsim.gates import CNOT, H

np.set_printoptions(precision=3, suppress=True)


@qsim.gate
def friend_measures(system: qsim.Qubit, record: qsim.Qubit) -> None:
    """The friend, inside the sealed laboratory, measures `system` and writes it down.

    One CNOT. That is the whole model, and it is not a simplification for the sake of
    the demo: a measurement *is* an interaction that correlates an apparatus with a
    system, and the smallest apparatus that can hold one bit is one qubit. The friend's
    notebook, retina and memory are all further qubits correlated with this one.
    """
    CNOT(system, record)

## 1. Inside the box: it looks exactly like collapse

Wigner puts his friend, a qubit `q`, and an apparatus in a sealed laboratory. The qubit
is prepared in $\lvert +\rangle$ — a superposition with a fifty-fifty chance of either
answer. The friend measures it.

Wrap that measurement in `qsim.within`. The scope applies its argument now, runs the body
eagerly, and undoes the argument on the way out — so *inside* the `with` block we are
standing next to the friend, after the measurement, and outside it we are Wigner, after
he has reversed the whole laboratory.

In [ ]:
box = Circuit(name="wigner", seed=42)
q = box.alloc("q")
friend = box.alloc("friend")

H(q)                                # the qubit: an even superposition
print("before the friend looks:", box.inspect.ket())

with qsim.within(friend_measures, q, friend):
    print()
    print("--- inside the box, after the friend has measured ---")
    print("the joint state    :", box.inspect.ket())
    print("q's density matrix :")
    print(np.round(box.inspect.reduced_density_matrix([q]), 12))
    print("q's Bloch vector   :", np.round(box.inspect.bloch_vector(q), 12))
    print("q's entropy        :", round(box.inspect.entanglement_entropy([q]), 12), "bits")

print()
print("--- outside, after Wigner reverses the laboratory ---")
print("the joint state    :", box.inspect.ket())
print("q's entropy        :", round(box.inspect.entanglement_entropy([q]), 12), "bits")

Inside the box, $\rho_q$ is $\mathrm{diag}(0.5, 0.5)$ with nothing off the diagonal. That
is *numerically identical* to the density matrix describing an honest classical mixture —
"the friend definitely saw 0, or definitely saw 1, with probability one half each, and we
have not been told which". There is no measurement, no experiment, no statistic that the
friend or anyone else confined to `q` can perform to tell the two apart. The friend is
entitled to say the qubit collapsed.

Here is that equality spelled out rather than asserted.

In [ ]:
# Build the "classical mixture" density matrix by hand: half of |0><0| plus half of
# |1><1|. np.outer(v, v.conj()) is the projector onto v — the outer product of the
# column vector with its own conjugate row.
ket0 = np.array([1.0, 0.0])
ket1 = np.array([0.0, 1.0])
mixture = 0.5 * np.outer(ket0, ket0.conj()) + 0.5 * np.outer(ket1, ket1.conj())

inside = Circuit(name="inside", seed=42)
iq = inside.alloc("q")
ifriend = inside.alloc("friend")
H(iq)
friend_measures(iq, ifriend)

print("q's state, with the friend still entangled:")
print(np.round(inside.inspect.reduced_density_matrix([iq]), 12))
print()
print("a classical coin flip between |0> and |1>:")
print(mixture)
print()
print("identical:", np.allclose(inside.inspect.reduced_density_matrix([iq]), mixture))

## 2. Outside the box: Wigner undoes the friend

But the global state is not a mixture. It is
$(\lvert 00\rangle + \lvert 11\rangle)/\sqrt2$ — a Bell state, pure, with zero entropy.
Every branch is still there, and a `CNOT` is a unitary like any other, so it can be run
backwards.

`friend_measures` is a `@qsim.gate` **block**, and `.adjoint()` turns any block into
another block that undoes it. Three spellings of the same physics follow; they produce
identical states, and which one reads best depends on what you are trying to say.

In [ ]:
def wigner_experiment(undo: str) -> float:
    """Run the friend's measurement, optionally undo it, then close the interferometer.

    Returns P(q = 0). If the friend's record really collapsed the qubit, no undoing is
    possible and this is 0.5 whatever we do. If it is a unitary, undoing restores the
    superposition and the final H sends it back to |0> with certainty.
    """
    qc = Circuit(name=f"wigner-{undo}", seed=42)
    wq = qc.alloc("q")
    wfriend = qc.alloc("friend")
    H(wq)

    if undo == "none":
        friend_measures(wq, wfriend)                      # record kept
    elif undo == "block-adjoint":
        friend_measures(wq, wfriend)
        friend_measures.adjoint()(wq, wfriend)            # ... and undone
    elif undo == "within":
        with qsim.within(friend_measures, wq, wfriend):
            pass                                          # record made and undone
    elif undo == "checkpoint":
        mark = qc.checkpoint()
        friend_measures(wq, wfriend)
        qc.rewind(mark)                                   # ... and rewound off the tape

    H(wq)                                                 # close the interferometer
    return float(np.real(qc.inspect.reduced_density_matrix([wq])[0, 0]))


for undo in ("none", "block-adjoint", "within", "checkpoint"):
    print(f"{undo:>16}:  P(q = 0) = {wigner_experiment(undo):.12f}")

$P(0) = 1$, three ways. The friend's record was made and unmade, and the qubit came out
of the laboratory in exactly the superposition it went in with — which a collapsed qubit
could not do, because a collapsed qubit would give the fair coin of the first row.

The three spellings say slightly different things about *who is doing the undoing*:

- `friend_measures.adjoint()(q, friend)` — Wigner applies a specific unitary, the inverse
  of the friend's apparatus. This is the physical story.
- `with qsim.within(friend_measures, q, friend):` — the record exists for the duration of
  a scope and is guaranteed to be undone on the way out. This is the story as a
  *bracket*: the friend's experience is a parenthesis in Wigner's description.
- `checkpoint` / `rewind` — Wigner does not name the unitary at all; he asks the tape what
  happened since a mark and replays it backwards. This is the story as an *undo button*,
  and it is the version that generalises to a laboratory full of apparatus whose exact
  unitary nobody wrote down.

They also differ in what they refuse. Try to `rewind` across an actual measurement, and
the tape says no — see [04 §7](../04-combinators.ipynb) and section 4 below.

## 3. The cut is not only movable, it is continuous

If "the collapse" happened at some definite moment, there should be a threshold — a point
at which the friend has looked *enough*. There is not. Replace the friend's `CNOT` with
`dephasing_coupling(q, friend, theta=t)`, which is a partial glance of adjustable
strength, and the interference dies smoothly.

In [ ]:
thetas = np.linspace(0.0, np.pi, 121)
p0_curve = []

for theta in thetas:
    qc = Circuit(name="partial-friend", seed=42)
    pq = qc.alloc("q")
    pfriend = qc.alloc("friend")
    H(pq)
    dephasing_coupling(pq, pfriend, theta=theta)     # the friend glances, by degrees
    H(pq)
    p0_curve.append(float(np.real(qc.inspect.reduced_density_matrix([pq])[0, 0])))

fig_curve, ax_curve = plt.subplots(figsize=(6.8, 3.8))

ax_curve.plot(thetas, p0_curve, lw=2.4, color="crimson", label="P(q = 0)")
ax_curve.plot(thetas, 0.5 * (1.0 + np.cos(thetas / 2)), "k--", lw=1.2,
              label=r"$\frac{1}{2}(1 + \cos(\theta/2))$")
ax_curve.axhline(0.5, color="gray", lw=0.8, ls=":")
ax_curve.set_xticks([0, np.pi / 2, np.pi], ["0", "π/2", "π"])
ax_curve.set_xlabel(r"$\theta$ — how hard the friend looked")
ax_curve.set_ylabel("P(q = 0) after recombining")
ax_curve.set_ylim(0.4, 1.15)
ax_curve.legend(fontsize=9)
ax_curve.set_title("no threshold anywhere: the cut slides")
fig_curve.tight_layout()

There is no kink, no step, no special angle. At $\theta = 0$ the friend has not looked and
interference is perfect; at $\theta = \pi$ the friend has a perfect record and
interference is gone; every value in between is a friend who half-looked and a fringe
pattern that is half there. Any answer to "at which $\theta$ did the collapse occur?"
would have to be drawn on a smooth curve with a ruler.

This is the same curve as [decoherence_dial](decoherence_dial.ipynb), which is the point:
**the friend is a dephasing environment with a name.** Nothing in the mathematics knows
that one of these qubits is conscious, in a laboratory, or writing in a notebook.

## 4. What is *not* movable: an actual measurement

`qsim` has one operation that is not a unitary, and it behaves completely differently.
`qc.measure()` picks a branch and discards the others — and there is no gate that brings a
discarded branch back.

In [ ]:
real = Circuit(name="real-measurement", seed=42)
rq = real.alloc("q")
rfriend = real.alloc("friend")
H(rq)
mark = real.checkpoint()

friend_measures(rq, rfriend)
reading = real.measure(rfriend)             # Wigner opens the box and looks too
print(f"the friend's record read: {reading}")

friend_measures.adjoint()(rq, rfriend)      # the same undo as before
H(rq)
p0_measured = float(np.real(real.inspect.reduced_density_matrix([rq])[0, 0]))
print(f"P(q = 0) after the same undo: {p0_measured:.12f}   (0.5 = no interference left)")
print()

try:
    real.rewind(mark)
except QsimError as err:
    print(err)

Two different refusals, and the difference matters. The undo *ran* — it is a legitimate
unitary and the library applied it without complaint — and it simply had nothing left to
work with. The `rewind` did not run at all: the tape can see that a measurement lies
between here and the mark, so it refuses before touching the state.

Side by side, then: the three protocols of this notebook, scored by the only number a
laboratory could actually report.

In [ ]:
bar_labels = ["record kept\n(friend looked)", "record undone\n(Wigner reversed it)",
              "friend really\nmeasured"]
bar_values = [wigner_experiment("none"), wigner_experiment("within"), p0_measured]

fig_bars, ax_bars = plt.subplots(figsize=(6.4, 3.6))
bars = ax_bars.bar(bar_labels, bar_values, width=0.55,
                   color=["crimson", "teal", "crimson"])
ax_bars.bar_label(bars, fmt="%.3f", padding=3)
ax_bars.axhline(0.5, color="gray", lw=0.8, ls=":")
ax_bars.set_ylabel("P(q = 0) after recombining")
ax_bars.set_ylim(0.0, 1.2)
ax_bars.set_title("only the middle bar had its record undone")
fig_bars.tight_layout()

The first and third bars agree — a kept record and a read record are equally fatal to
interference, which is the sense in which decoherence "is" measurement for all practical
purposes. But only the middle bar could be undone, and only the third one severs the tape.

That is the honest boundary of what this notebook shows:

- **Decoherence explains why the friend sees a definite-looking world**, and why we do
  not encounter fringes from macroscopic superpositions. It is entirely mechanical and
  there is nothing mysterious in it.
- **It does not explain why the friend sees *one* outcome.** Both branches are still in
  the state; the friend in branch 0 and the friend in branch 1 are equally real
  descriptions in that mathematics. The step from "the state contains both" to "I saw 0"
  is the measurement problem, and this simulator does not solve it. What it does is make
  the problem precise enough to stop being vague.

## 5. Deferred measurement, and why any of this matters practically

There is a theorem behind all of this, and it is boringly useful: **any mid-circuit
measurement can be replaced by a `CNOT` onto a fresh qubit that is then left alone**, with
identical final statistics. Section 1 is that theorem's proof in the smallest possible
case. Compilers use it constantly — it is why a quantum algorithm written with measurement
and classical feedback can be run on hardware that only does unitaries.

The one thing deferral *loses* is exactly the thing section 2 gains: with the measurement
deferred, there is no classical outcome to condition on, and the branches remain
recoverable. Which is either an engineering convenience or the deepest problem in the
foundations of physics, depending on the day.

## Where to go next

- **[quantum_eraser](quantum_eraser.ipynb)**: the same undo without the philosophy, plus
  the reason the laboratory is not like section 2 (the friend has $10^{23}$ degrees of
  freedom, and undoing all of them is a budget problem, not a physics problem).
- **[einselection](einselection.ipynb)**: which alternatives the friend ends up seeing,
  and why they are the position-like ones.
- **[04 — Programs made of gates](../04-combinators.ipynb)** §7: coherent records versus
  classical outcomes, stated as a property of the tape.

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. From inside, the friend's record makes q numerically identical to a coin flip.
assert np.allclose(inside.inspect.reduced_density_matrix([iq]), mixture, atol=1e-12)
assert np.allclose(inside.inspect.bloch_vector(iq), (0.0, 0.0, 0.0), atol=1e-12)
assert np.isclose(inside.inspect.entanglement_entropy([iq]), 1.0)

# 2. ... while the joint state is pure the whole time. Nothing collapsed.
assert np.isclose(inside.inspect.entanglement_entropy(list(inside.qubits)), 0.0, atol=1e-12)
assert np.isclose(inside.inspect.norm(), 1.0)

# 3. Undoing the record restores interference exactly, in all three spellings.
assert np.isclose(wigner_experiment("none"), 0.5, atol=1e-12)
for spelling in ("block-adjoint", "within", "checkpoint"):
    assert np.isclose(wigner_experiment(spelling), 1.0, atol=1e-12), spelling

# 4. `within` really does put the state back where it started.
before = Circuit(name="check-within", seed=42)
bq = before.alloc("q")
bfriend = before.alloc("friend")
H(bq)
snapshot = before.inspect.state_vector()
with qsim.within(friend_measures, bq, bfriend):
    assert np.isclose(before.inspect.entanglement_entropy([bq]), 1.0)
assert np.allclose(before.inspect.state_vector(), snapshot, atol=1e-12)

# 5. The cut is continuous: P(0) = (1 + cos(theta/2)) / 2, with no threshold.
assert np.allclose(p0_curve, 0.5 * (1.0 + np.cos(thetas / 2)), atol=1e-12)

# 6. A real measurement cannot be undone, and the tape refuses to try.
assert np.isclose(p0_measured, 0.5, atol=1e-12)
try:
    real.rewind(mark)
except QsimError as err:
    assert "measurement" in str(err)
else:
    raise AssertionError("rewinding across a measurement should have raised")

print("all assertions passed")